# 11.05 · Subagents and delegation

**Module 11 · Deep Agents** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 2–2.5 h |
| 📋 Prerequisites | Lessons 11.01–11.04; Module 09 (subagents pattern, Project 09 support desk). |
| 🔑 Keys | none for 🧪 cells · 💸 cell ≈ 6–12K tokens |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Define** custom subagents (`name`, `description`, `system_prompt`, `tools`, `model`, `response_format`, `permissions`) and wrap an existing agent as a `CompiledSubAgent`.
2. **Explain** context isolation: what a subagent sees, what comes back, and why that saves the coordinator's context.
3. **Run** several subagents in parallel from one model turn.
4. **Rebuild** the Project 09 technical specialist as a deep-agent subagent with least-privilege tools.
5. **Decide** when to delegate and when one agent is enough.

## 📚 Official reading (verified 2026-10-16, `deepagents` 0.7.26)

| Page | Read it for |
|---|---|
| [Subagents](https://docs.langchain.com/oss/python/deepagents/subagents) | Spec fields, the general-purpose subagent, `CompiledSubAgent`, structured output, forking (beta, Lesson 11.12) |
| [Multi-agent: subagents pattern](https://docs.langchain.com/oss/python/langchain/multi-agent/subagents) | The pattern from Module 09 |
| [Permissions: subagent permissions](https://docs.langchain.com/oss/python/deepagents/permissions) | Inherit by default, replace with a spec's `permissions` |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "deepagents>=0.7.23,<0.8" "langchain-google-genai>=4.4.1,<5" "langchain-groq>=1.1.3,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("deepagents", "langchain",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/11-deep-agents/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GOOGLE_API_KEY', 'GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm11-subagents'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.deep import deep_key_ready, deep_model_spec, fallback_specs

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")

DEEP_MODEL = deep_model_spec("main")              # MASTERY_DEEP_MODEL (course default: Gemini free tier)
SUBAGENT_MODEL = deep_model_spec("subagent")      # MASTERY_DEEP_SUBAGENT_MODEL
HAVE_DEEP_KEY = deep_key_ready(DEEP_MODEL)        # the key for THAT provider (GOOGLE_API_KEY for google_genai:)
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m11-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("deep model:", DEEP_MODEL, "| subagents:", SUBAGENT_MODEL)
print("fallbacks :", fallback_specs(DEEP_MODEL) or "none (optional: MASTERY_FALLBACK_MODELS in .env)")
if not HAVE_DEEP_KEY:
    print(f"\n⚠️ No key for {DEEP_MODEL.split(':')[0]}: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

In [ ]:
# 🧪 Helpers: a model turn with one or more tool calls, a user turn, the task results of a run.
from langchain_core.messages import AIMessage

from mastery.deep import build_deep_agent, run_stats
from mastery.testing import ScriptedChatModel


def calls(*items) -> AIMessage:
    """calls(("task", {...}, "t1"), ...): one model turn with several tool calls (they run in parallel)."""
    return AIMessage("", tool_calls=[{"name": n, "args": a, "id": i} for n, a, i in items])


def user(text: str) -> dict:
    return {"messages": [{"role": "user", "content": text}]}


def task_results(result) -> list[str]:
    return [m.content for m in result["messages"] if m.type == "tool" and m.name == "task"]

---
## Part 1 · What a subagent is in Deep Agents

The `task` tool starts a subagent with a **fresh conversation**: only the coordinator's `description` goes in, only the subagent's **final message** comes back. Its tool calls, retries and raw results never enter the coordinator's context (Lesson 11.01's mini-harness did the same). Files are the exception: with `StateBackend` they are shared (Lesson 11.03).

| Spec field | Meaning |
|---|---|
| `name`, `description` | How the coordinator picks it (the descriptions are listed in the `task` tool: write them like tool docstrings) |
| `system_prompt` | Its own instructions: always write one, it is all the subagent knows about its role |
| `tools` | Its own tools; default: the coordinator's tools |
| `model` | Its own model; default: the coordinator's (the course uses `MASTERY_DEEP_SUBAGENT_MODEL`) |
| `response_format` | Structured output: the coordinator receives JSON |
| `permissions`, `interrupt_on`, `middleware`, `skills` | Per-subagent guardrails and extras |
| `runnable` (instead of the above) | A `CompiledSubAgent`: any agent or graph you already built |

A **general-purpose** subagent (same tools and model as the coordinator) is added unless you disable it (Lesson 11.02) or define one named `general-purpose`.

## Part 2 · The Project 09 specialist, as a subagent

In [ ]:
# 🧪 Custom subagents: a dict spec each. The Project 09 technical specialist, rebuilt as a deep-agent subagent.
from mastery.agents.support import SupportDesk, make_support_tools

desk_tools = make_support_tools(SupportDesk.load())          # Module 09 tools: billing / technical / account
tech_model = ScriptedChatModel(responses=[calls(("check_service_status", {"service": "api"}, "s1")),
                                          AIMessage("The API is operational; no incident.")])
technical = {
    "name": "technical",
    "description": "Orbita Cloud service status and FAQ questions. Give the customer's exact question.",
    "system_prompt": "You are Orbita Cloud's technical specialist. Use your tools; answer in two sentences.",
    "tools": desk_tools["technical"],                         # only ITS tools: least privilege
    "model": tech_model,                                      # LIVE: deep_model("subagent")
}
main = ScriptedChatModel(responses=[
    calls(("task", {"subagent_type": "technical", "description": "Customer asks: is the API down right now?"}, "t1")),
    AIMessage("Good news: the API is operational.")])
desk = build_deep_agent(main, subagents=[technical], system_prompt="You are Orbita Cloud's support coordinator.",
                        fallbacks=False)
out = desk.invoke(user("Is your API down?"))
print("task →", task_results(out))
print("coordinator →", out["messages"][-1].content)
print("coordinator tools:", main.bound_tools, "| specialist tools:", tech_model.bound_tools)

**You should now see** the specialist's answer coming back through `task`, and two tool lists: the coordinator has the file tools and `task`; the specialist has **only** `search_faq` and `check_service_status`. In Project 09 you built this with `task_tool`; here the harness gives you the dispatch, the isolation and the shared files.

## Part 3 · Parallel delegation and structured results

In [ ]:
# 🧪 Three subagent kinds, called IN PARALLEL from one model turn.
from langchain.agents import create_agent
from pydantic import BaseModel, Field


class Finding(BaseModel):
    """One claim with its evidence (Project 11's researchers return these)."""
    claim: str
    evidence: str = Field(description="A short quote that supports the claim.")
    source: str


researcher = {"name": "researcher", "description": "Returns ONE structured finding with evidence.",
              "system_prompt": "Return exactly one Finding.", "tools": [], "response_format": Finding,
              "model": ScriptedChatModel(responses=[calls(("Finding", {
                  "claim": "Checkpointers save state after every step", "evidence": "saves a checkpoint at every super-step",
                  "source": "https://docs.langchain.com/oss/python/langgraph/persistence"}, "f1"))])}
billing_agent = create_agent(ScriptedChatModel(responses=[AIMessage("Invoice INV-2001 is paid.")]),
                             tools=[], name="billing")
billing = {"name": "billing", "description": "Invoices and payments.", "runnable": billing_agent}   # CompiledSubAgent

main = ScriptedChatModel(responses=[
    calls(("task", {"subagent_type": "technical", "description": "Is the API up?"}, "t1"),
          ("task", {"subagent_type": "researcher", "description": "One fact about checkpointers"}, "t2"),
          ("task", {"subagent_type": "billing", "description": "Is INV-2001 paid?"}, "t3")),
    AIMessage("All three answers are in.")])
tech_model.reset()
team = build_deep_agent(main, subagents=[technical, researcher, billing], fallbacks=False)
for text in task_results(team.invoke(user("Three questions at once."))):
    print("task →", text[:110])

**What just happened?** One model turn with three `task` calls: the tool node runs them in parallel, so the wall-clock time is the slowest subagent, not the sum. The researcher returned **JSON** (its `response_format`), which the coordinator can cite without re-parsing prose. `billing` was an ordinary `create_agent` wrapped with `runnable`: anything that takes `messages` and returns `messages` can be a subagent.

> ⚠️ **Production note: parallel subagents multiply rate-limit pressure.** Three subagents at once are three bursts of calls against one quota. In production: a shared rate limiter per provider (the course's `deep_model()` gives each model its own), call budgets per subagent (`middleware=` in the spec), and limits on how many tasks one turn may start.

## Part 4 · 💸 Delegation on a real model

> 💸 Calls your deep model (`MASTERY_DEEP_MODEL`, default Gemini free tier). Rate-limited to one call every 4 s and capped per run. Without the key it prints `⏭️ Skipped`.

In [ ]:
# 💸 LIVE: the coordinator delegates to the technical specialist on your models. ≈ 6–12K tokens.
if HAVE_DEEP_KEY:
    from mastery.deep import deep_model

    live_tech = {**technical, "model": deep_model("subagent")}
    live_desk = build_deep_agent(subagents=[live_tech], model_calls=8, tool_calls=12,
                                 system_prompt="You are Orbita Cloud's support coordinator. Delegate technical "
                                               "questions to the `technical` subagent; never guess service status.")
    live = live_desk.invoke(user("Hi, is the Orbita API having an outage? My deploys are failing."))
    print("delegated:", [m.content[:120] for m in live["messages"] if m.type == "tool" and m.name == "task"] or "no")
    print("answer   :", live["messages"][-1].text[:400])
    print("stats    :", run_stats(live))
else:
    print(f"⏭️ Skipped: no key for {DEEP_MODEL} (see the setup cell).")

**You should now see** whether the coordinator delegated (`delegated:`), the answer, and the coordinator's stats. Real models sometimes answer without delegating: that is a **bad delegation** failure, one of the categories you will count in Lesson 11.11. Make the subagent's `description` and the coordinator's prompt explicit about **when** to delegate.

## Part 5 · When to delegate

| Delegate when | Don't, when |
|---|---|
| The sub-task produces lots of intermediate noise (searches, retries) | The answer is one tool call away |
| It needs different tools, permissions or a different model | The coordinator needs the sub-task's full reasoning, not a summary |
| Independent sub-tasks can run in parallel | Every delegation adds a fixed prompt + tool schemas: on a free tier that's real tokens |

## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| Subagent asks "which customer?" | It only sees the `description`, not the chat | Tell the coordinator to put every needed fact in the description |
| Coordinator never delegates | Vague subagent `description` | Say what it does, when to use it, what to pass |
| Subagent has tools it shouldn't | `tools` omitted, so it inherited the coordinator's | Always set `tools` explicitly |
| Two subagents overwrite one file | Shared `StateBackend` | Separate paths per subagent (`/research/web/`, `/research/docs/`) |

## 🏋️ Exercises

1. **(Easy)** Add the Project 09 `account` specialist as a fourth subagent with `desk_tools["account"]`.
2. **(Medium)** Give the technical subagent read-only file permissions (`permissions=[...]` in its spec) and show it can't write.
3. **(Medium)** Disable the general-purpose subagent with a harness profile (Lesson 11.02) and check `task`'s choices.
4. **(Stretch, 💸)** Run Part 4 five times and count how often the coordinator delegates. Report the rate with a Wilson interval.

<details>
<summary><b>Show solutions</b></summary>

**1.**
```python
account = {"name": "account", "description": "Password resets, unlocks, email changes.",
           "system_prompt": "You are Orbita Cloud's account specialist.", "tools": desk_tools["account"],
           "model": ScriptedChatModel(responses=[AIMessage("Reset link sent.")])}
```

**2.**
```python
from deepagents import FilesystemPermission
ro_tech = {**technical, "permissions": [FilesystemPermission(operations=["write"], paths=["/**", "/**/.*"], mode="deny")]}
```
Script the specialist to call `write_file` and read the `permission denied` result.

**3.** With `GeneralPurposeSubagentProfile(enabled=False)` the `task` tool's `subagent_type` choices are only your custom subagents.

**4.**
```python
from mastery.evals.stats import wilson_interval
runs = [live_desk.invoke(user("Is the API down?")) for _ in range(5)]
hits = sum(any(m.type == "tool" and m.name == "task" for m in r["messages"]) for r in runs)
print(hits, wilson_interval(hits, 5))
```
</details>

## 🔎 Docs drill (5 min)

If a subagent spec omits `tools` and `model`, what does the subagent get? And what does a subagent spec's `permissions` do to the parent's rules?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Subagents](https://docs.langchain.com/oss/python/deepagents/subagents) and [Permissions → Subagent permissions](https://docs.langchain.com/oss/python/deepagents/permissions).
2. Omitted `tools` / `model`: the coordinator's tools and model.
3. Permissions are inherited by default; a spec's `permissions` **replaces** the parent's rules entirely.

</details>

## ✅ Checkpoint

- [ ] I can define subagents with their own prompt, tools, model and structured output, and wrap an existing agent.
- [ ] I can explain what crosses the `task` boundary (description in, final message out, shared files).
- [ ] I run independent sub-tasks in parallel and know what that does to rate limits.
- [ ] I give every subagent explicit, minimal tools.
- [ ] Restart & Run All passes without keys; outputs cleared before saving.

## 🔎 Research checkpoint

Read [Subagents](https://docs.langchain.com/oss/python/deepagents/subagents): is `mode="fork"` still beta (2026-10-16: yes, needs ≥ 0.7.13)? What does a forked subagent see that an isolated one doesn't? Log the answer with versions in [`docs/research-log.md`](../../docs/research-log.md); Lesson 11.12 (briefing F1) builds on it.

## Summary

- A subagent = fresh context in, final message out; files are shared.
- Specs set prompt, tools, model, structured output and guardrails; `runnable` wraps any agent.
- Parallel `task` calls cut latency and multiply rate-limit pressure.
- Delegate for isolation, different privileges or parallelism, not by default.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 11-06-context-engineering-long-runs.ipynb** → [`11-06-context-engineering-long-runs.ipynb`](11-06-context-engineering-long-runs.ipynb).

[Module 11 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)